# Testing the class `GeneralizedAlgebraicKnot`

## Setup

In [1]:
import notebook_setup
import ipytest
import pytest

ipytest.autoconfig(raise_on_error=True)

The ipytest module is not an IPython extension.
INFO: __init__.py: current_file_path=/Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/gaknot_lib/__init__.py
INFO: __init__.py: calculated path=/Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: safe_import: checking for /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/gaknot_lib/signature.sage
INFO: 

import_sage called with arguments:
	module_name: signature
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: Checking for sage file at: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/gaknot_lib/signature.sage
INFO: safe_import: checking for /Users/wojtek/Library/CloudStorag

In [2]:
%preparse gaknot_lib
from gaknot_lib.gaknot import GeneralizedAlgebraicKnot

INFO: 

import_sage called with arguments:
	module_name: gaknot
	package: gaknot_lib
	path: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function
INFO: Checking for sage file at: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/gaknot_lib/gaknot.sage
Successfully preparsed and reloaded: gaknot


## Tests

### Basic Functionality
Verifies the fundamental construction of the `GeneralizedAlgebraicKnot` class, including string representation and internal description storage.

This test verifies that the `GeneralizedAlgebraicKnot` class can be initialized with various knot descriptions and that its human-readable string representation correctly reflects the construction (signs, cabling parameters, and connected sum structure).

In [3]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc, expected_str", [
    ([(1, [(2, 3)])], "T(2,3)"),
    ([(-1, [(2, 3)])], "-T(2,3)"),
    ([(1, [(2, 3), (2, 5)])], "T(2,3; 2,5)"),
    ([(-1, [(2, 3), (2, 5)])], "-T(2,3; 2,5)"),
    ([(1, [(2, 3)]), (1, [(3, 4)])], "T(2,3) # T(3,4)"),
    ([(1, [(2, 3)]), (-1, [(3, 4)])], "T(2,3) # -T(3,4)"),
    ([(1, [(2, 3), (6, 5)]), (-1, [(2, 3)])], "T(2,3; 6,5) # -T(2,3)"),
    ([(1, [(2, 3), (6, 5)]), (-1, [(2, 3), (6, 7)])], "T(2,3; 6,5) # -T(2,3; 6,7)"),
    ([(1, [(2, 3)]), (1, [(3, 5)]), (1, [(5, 7)])], "T(2,3) # T(3,5) # T(5,7)"),
    ([(1, [(2, 3), (6, 5), (30, 7)])], "T(2,3; 6,5; 30,7)")
])
def test_gaknot_basic_functionality(desc, expected_str):
    knot = GeneralizedAlgebraicKnot(desc)
    assert str(knot) == expected_str
    assert knot.description == desc

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_basic_functionality[desc0-T(2,3)] PASSED  [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_basic_functionality[desc1--T(2,3)] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_basic_functionality[desc2-T(2,3; 2,5)] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_basic_functionality[desc3--T(2,3; 2,5)] PASSED [ 40%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_basic_functionality[desc4-T(2,3) # T(3,4)] PASSED [ 50%]
t_3a6e8db1ae7e4860b0

### Validation
Ensures that the class correctly validates its input, catching errors such as non-coprime parameters, non-positive parameters, invalid signs, and incorrect structural formatting using 10 distinct test cases.

This test ensures that the `GeneralizedAlgebraicKnot` class performs rigorous validation of its input description. It checks for a wide range of error conditions, including incorrect data types, malformed structures, invalid signs, non-positive cabling parameters, and parameters that are not relatively prime.

In [4]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc, error_type, match", [
    # Not a list/tuple
    ("not a list", TypeError, "must be a list or tuple"),
    # Element not a pair
    ([(1, [(2, 3)]), (1,)], ValueError, "must be a pair"),
    # Bad sign
    ([(2, [(2, 3)])], ValueError, "Sign at index 0 must be 1 or -1"),
    # Knot description not a list
    ([(1, "not a list")], TypeError, "must be a list or tuple"),
    # Cable parameter not a pair
    ([(1, [(2, 3, 4)])], ValueError, r"must be a pair \(p, q\)"),
    # Parameters not integers
    ([(1, [(2.5, 3)])], TypeError, "must be integers"),
    # p <= 1
    ([(1, [(1, 3)])], ValueError, "must be > 1"),
    # q <= 1
    ([(1, [(3, 0)])], ValueError, "must be > 1"),
    # Not relatively prime (2, 4)
    ([(1, [(2, 4)])], ValueError, "relatively prime"),
    # Not relatively prime (6, 9) in a deeper layer
    ([(1, [(2, 3), (6, 9)])], ValueError, "relatively prime")
])
def test_gaknot_validation_parametric(desc, error_type, match):
    with pytest.raises(error_type, match=match):
        GeneralizedAlgebraicKnot(desc)

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_validation_parametric[not a list-TypeError-must be a list or tuple] PASSED [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_validation_parametric[desc1-ValueError-must be a pair] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_validation_parametric[desc2-ValueError-Sign at index 0 must be 1 or -1] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_validation_parametric[desc3-TypeError-must be a list or tuple] PASSED [ 40%]
t_3a6e8db1ae7e4860b

### Algebraic Operations
Tests the overloaded algebraic operators for connected sums (`+`), concordance inverses (`-`), and differences (`-`).

This test confirms that the overloaded algebraic operators (`+`, `-`, and unary `-`) correctly perform connected sums, concordance inverses, and differences of knots, respectively. It ensures the resulting knot descriptions and string representations are accurate.

In [5]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc1, desc2, expected_sum, expected_neg1, expected_diff", [
    ([(1, [(2, 3)])], [(1, [(3, 4)])], "T(2,3) # T(3,4)", "-T(2,3)", "T(2,3) # -T(3,4)"),
    ([(1, [(2, 5)])], [(1, [(2, 3), (6, 5)])], "T(2,5) # T(2,3; 6,5)", "-T(2,5)", "T(2,5) # -T(2,3; 6,5)"),
    ([(-1, [(2, 3)])], [(1, [(3, 4)])], "-T(2,3) # T(3,4)", "T(2,3)", "-T(2,3) # -T(3,4)"),
    ([(1, [(2, 3), (6, 5)])], [(-1, [(2, 3), (6, 5)])], "T(2,3; 6,5) # -T(2,3; 6,5)", "-T(2,3; 6,5)", "T(2,3; 6,5) # T(2,3; 6,5)"),
    ([(1, [(2, 3)]), (1, [(3, 4)])], [(1, [(4, 5)])], "T(2,3) # T(3,4) # T(4,5)", "-T(2,3) # -T(3,4)", "T(2,3) # T(3,4) # -T(4,5)"),
    ([(1, [(2, 3)])], [(1, [(3, 4)]), (1, [(4, 5)])], "T(2,3) # T(3,4) # T(4,5)", "-T(2,3)", "T(2,3) # -T(3,4) # -T(4,5)"),
    ([(-1, [(2, 3), (2, 5)])], [(-1, [(3, 4), (3, 5)])], "-T(2,3; 2,5) # -T(3,4; 3,5)", "T(2,3; 2,5)", "-T(2,3; 2,5) # T(3,4; 3,5)"),
    ([(1, [(2, 3)])], [(1, [(2, 3)])], "T(2,3) # T(2,3)", "-T(2,3)", "T(2,3) # -T(2,3)"),
    ([(1, [(2, 3), (2, 5), (2, 7)])], [(1, [(3, 4)])], "T(2,3; 2,5; 2,7) # T(3,4)", "-T(2,3; 2,5; 2,7)", "T(2,3; 2,5; 2,7) # -T(3,4)"),
    ([(1, [(5, 7)])], [(-1, [(5, 7)])], "T(5,7) # -T(5,7)", "-T(5,7)", "T(5,7) # T(5,7)")
])
def test_gaknot_algebraic_operations_parametric(desc1, desc2, expected_sum, expected_neg1, expected_diff):
    knot1 = GeneralizedAlgebraicKnot(desc1)
    knot2 = GeneralizedAlgebraicKnot(desc2)

    sum_knot = knot1 + knot2
    assert str(sum_knot) == expected_sum

    neg_knot1 = -knot1
    assert str(neg_knot1) == expected_neg1

    diff_knot = knot1 - knot2
    assert str(diff_knot) == expected_diff

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_algebraic_operations_parametric[desc10-desc20-T(2,3) # T(3,4)--T(2,3)-T(2,3) # -T(3,4)] PASSED [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_algebraic_operations_parametric[desc11-desc21-T(2,5) # T(2,3; 6,5)--T(2,5)-T(2,5) # -T(2,3; 6,5)] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_algebraic_operations_parametric[desc12-desc22--T(2,3) # T(3,4)-T(2,3)--T(2,3) # -T(3,4)] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_algebraic_operat

### Alexander Polynomial
Verifies the computation of the Alexander polynomial for both simple torus knots and their connected sums.

This test verifies the computation of the Alexander polynomial for various generalized algebraic knots. It ensures that the result matches the product of the Alexander polynomials of the individual connected sum components.

In [6]:
%%ipytest -vv -W ignore::DeprecationWarning

from sage.all import PolynomialRing, ZZ
R_alex = PolynomialRing(ZZ, 't')
t_alex = R_alex.gen()

@pytest.mark.parametrize("desc, expected_poly", [
    ([(1, [(2, 3)])], t_alex**2 - t_alex + 1),
    ([(1, [(2, 5)])], t_alex**4 - t_alex**3 + t_alex**2 - t_alex + 1),
    ([(1, [(3, 4)])], t_alex**6 - t_alex**5 + t_alex**3 - t_alex + 1),
    ([(-1, [(2, 3)])], t_alex**2 - t_alex + 1),
    ([(1, [(2, 3)]), (1, [(2, 3)])], (t_alex**2 - t_alex + 1)**2),
    ([(1, [(2, 3)]), (-1, [(2, 3)])], (t_alex**2 - t_alex + 1)**2),
    ([(1, [(2, 3), (2, 5)])], (t_alex**4 - t_alex**2 + 1) * (t_alex**4 - t_alex**3 + t_alex**2 - t_alex + 1)),
    ([(1, [(2, 5), (2, 3)])], (t_alex**8 - t_alex**6 + t_alex**4 - t_alex**2 + 1) * (t_alex**2 - t_alex + 1)),
    ([(1, [(2, 3)]), (1, [(2, 5)])], (t_alex**2 - t_alex + 1) * (t_alex**4 - t_alex**3 + t_alex**2 - t_alex + 1)),
    ([(1, [(2, 3), (2, 5)]), (1, [(3, 2)])], (t_alex**4 - t_alex**2 + 1) * (t_alex**4 - t_alex**3 + t_alex**2 - t_alex + 1) * (t_alex**2 - t_alex + 1))
])
def test_gaknot_alexander_polynomial_parametric(desc, expected_poly):
    knot = GeneralizedAlgebraicKnot(desc)
    assert knot.alexander_polynomial() == expected_poly

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_alexander_polynomial_parametric[desc0-expected_poly0] PASSED [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_alexander_polynomial_parametric[desc1-expected_poly1] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_alexander_polynomial_parametric[desc2-expected_poly2] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_alexander_polynomial_parametric[desc3-expected_poly3] PASSED [ 40%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_alexander_

### Container Behavior
Tests that the class behaves like a container, supporting length checks, indexing, and slicing.

This test confirms that the `GeneralizedAlgebraicKnot` class implements the expected container protocols. It verifies `len()`, indexing (both positive and negative), slicing, and ensuring that out-of-range indexing raises an `IndexError`.

In [7]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc, test_fn", [
    # len tests
    ([(1, [(2, 3)])], lambda k: len(k) == 1),
    ([(1, [(2, 3)]), (1, [(3, 4)])], lambda k: len(k) == 2),
    ([(1, [(2, 3)]), (1, [(3, 4)]), (1, [(4, 5)])], lambda k: len(k) == 3),
    # indexing tests
    ([(1, [(2, 3)]), (-1, [(3, 4)])], lambda k: str(k[0]) == "T(2,3)"),
    ([(1, [(2, 3)]), (-1, [(3, 4)])], lambda k: str(k[1]) == "-T(3,4)"),
    ([(1, [(2, 3)]), (-1, [(3, 4)])], lambda k: str(k[-1]) == "-T(3,4)"),
    # slicing tests
    ([(1, [(2, 3)]), (1, [(3, 4)]), (1, [(4, 5)])], lambda k: str(k[0:2]) == "T(2,3) # T(3,4)"),
    ([(1, [(2, 3)]), (1, [(3, 4)]), (1, [(4, 5)])], lambda k: len(k[1:]) == 2),
    ([(1, [(2, 3)]), (1, [(3, 4)]), (1, [(4, 5)])], lambda k: str(k[:1]) == "T(2,3)"),
    # exception test
    ([(1, [(2, 3)])], lambda k: pytest.raises(IndexError, k.__getitem__, 10))
])
def test_gaknot_container_behavior_parametric(desc, test_fn):
    knot = GeneralizedAlgebraicKnot(desc)
    result = test_fn(knot)
    if result is not None:
        assert result

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_container_behavior_parametric[desc0-<lambda>] PASSED [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_container_behavior_parametric[desc1-<lambda>] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_container_behavior_parametric[desc2-<lambda>] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_container_behavior_parametric[desc3-<lambda>] PASSED [ 40%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_container_behavior_parametric[desc4-<lambd

### Type Verification
Ensures that the knot classification methods correctly identify positive torus knots, negative torus knots, and iterated torus knots.

This test verifies the knot classification methods (`is_positive_torus_knot`, `is_negative_torus_knot`, and `is_iterated_torus_knot`). It ensures that these methods correctly identify the nature of the knot represented by the `GeneralizedAlgebraicKnot` object.

In [8]:
%%ipytest -vv -W ignore::DeprecationWarning

@pytest.mark.parametrize("desc, expected_results", [
    # Positive Torus Knot
    ([(1, [(2, 3)])], {'pos': True, 'neg': False, 'it': True}),
    # Negative Torus Knot
    ([(-1, [(2, 3)])], {'pos': False, 'neg': True, 'it': False}),
    # Positive Iterated Torus Knot (not basic)
    ([(1, [(2, 3), (2, 5)])], {'pos': False, 'neg': False, 'it': True}),
    # Connected sum (not a single torus/iterated knot)
    ([(1, [(2, 3)]), (1, [(3, 4)])], {'pos': False, 'neg': False, 'it': False}),
    # Single positive torus knot (different parameters)
    ([(1, [(3, 5)])], {'pos': True, 'neg': False, 'it': True}),
    # Single negative torus knot (different parameters)
    ([(-1, [(3, 5)])], {'pos': False, 'neg': True, 'it': False}),
    # Iterated torus knot with deeper layering
    ([(1, [(2, 3), (6, 5), (30, 7)])], {'pos': False, 'neg': False, 'it': True}),
    # Connected sum with negative component
    ([(1, [(2, 3)]), (-1, [(2, 3)])], {'pos': False, 'neg': False, 'it': False}),
    # Single positive torus knot (another one)
    ([(1, [(2, 7)])], {'pos': True, 'neg': False, 'it': True}),
    # Single negative iterated torus knot (not currently checked by a specific single method in tests, but let's see)
    ([(-1, [(2, 3), (2, 5)])], {'pos': False, 'neg': False, 'it': False})
])
def test_gaknot_type_verification_parametric(desc, expected_results):
    knot = GeneralizedAlgebraicKnot(desc)
    assert knot.is_positive_torus_knot() == expected_results['pos']
    assert knot.is_negative_torus_knot() == expected_results['neg']
    assert knot.is_iterated_torus_knot() == expected_results['it']

======================================= test session starts ========================================
platform darwin -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/anaconda3/envs/sage_env/bin/python3
cachedir: .pytest_cache
rootdir: /Users/wojtek/Library/CloudStorage/GoogleDrive-w.politarczyk@uw.edu.pl/My Drive/mat/git-projects/signature_function/notebooks
plugins: anyio-4.12.1, nbmake-1.5.5, langsmith-0.7.9
collecting ... collected 10 items

t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_type_verification_parametric[desc0-expected_results0] PASSED [ 10%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_type_verification_parametric[desc1-expected_results1] PASSED [ 20%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_type_verification_parametric[desc2-expected_results2] PASSED [ 30%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_type_verification_parametric[desc3-expected_results3] PASSED [ 40%]
t_3a6e8db1ae7e4860b0ed33dfadb5d626.py::test_gaknot_type_verif